# Topic 21 — Decision: when NOT to use an LLM

Every feature asks the same question, and the honest answer is often **"no LLM"**.
This notebook runs the hand-written `decide.py` framework on three realistic
features:

| feature | candidates | likely answer |
| --- | --- | --- |
| invoice field extraction | regex parser / small LLM / large LLM | regex — until the schema drifts |
| support-ticket routing | classifier / generation | classifier |
| FAQ answering | lookup table / embedding retrieval / RAG | lookup — until questions vary |

The framework is always the same: **constraints → candidates → trade-offs →
justified choice + sensitivity ("what flips it")**.

Everything is offline and deterministic.

In [ ]:
import sys

sys.path.insert(0, '.')  # run from the implementation/ folder

from decide import (
    DETERMINISTIC_KINDS, EXTRACTION_CRITERIA, KIND_CLASSIFIER, KIND_DETERMINISTIC,
    KIND_LARGE_LLM, KIND_LOOKUP, KIND_REGEX, KIND_SMALL_LLM, LLM_KINDS,
    Candidate, Criterion, Delta, TASK_TYPE_CRITERIA, apply_hard_constraints,
    choose_extraction, choose_faq, choose_task_type, cost_constraint, decide,
    deterministic_first, exactness_constraint, flip_thresholds, latency_constraint,
    never_llm_reason, offline_constraint, score_candidates, sensitivity,
    verifiability_gate,
)

print('module loaded')

## 0. The building blocks: hard constraints eliminate, criteria rank

A hard constraint is pass/fail — a 3s p99 is not "a low latency score". Criteria
are weighted preferences over the survivors.

In [ ]:
code = Candidate('code', KIND_DETERMINISTIC, {'quality': 0.70, 'cost': 0.55},
                 latency_ms=3.0, cost_usd=0.0, exact=True, verifiable=True)
llm = Candidate('llm', KIND_SMALL_LLM, {'quality': 0.98, 'cost': 0.20},
                latency_ms=900.0, cost_usd=0.0004, requires_network=True, exact=False, verifiable=True)
criteria = [Criterion('quality', 0.70), Criterion('cost', 0.30)]

print('without constraints, the soft scores pick:', score_candidates([code, llm], criteria).order())
print('deterministic-first candidates:', [c.name for c in deterministic_first([code, llm])])

for label, constraints in [
    ('latency <= 50ms', [latency_constraint(50.0)]),
    ('cost <= $0.00/request', [cost_constraint(0.0)]),
    ('offline / no network', [offline_constraint()]),
    ('exact output required', [exactness_constraint()]),
]:
    result = apply_hard_constraints([code, llm], constraints)
    print(f'{label:24s} -> survivors {[c.name for c in result.survivors]}, llm fails {result.reason_for("llm")!r}')

## 1. Invoice field extraction — the classic flip

Input is text, output is structured, and the format is known. Deterministic-first
says: a parser. Until the format drifts.

In [ ]:
for label, kwargs in [
    ('known format, stable', dict(known_format=True)),
    ('known format, drifting', dict(known_format=True, format_stability='drifting')),
    ('no known format', dict(known_format=False)),
]:
    decision = choose_extraction('INV-1042 | 2026-04-18 | 250.00 USD', **kwargs)
    print('=' * 72)
    print(label.upper())
    for score in decision.ranked.scores:
        print(f'  {score.candidate.name:22s} {score.total:.3f}  '
              f'latency {score.candidate.latency_ms:7.1f}ms  ${score.candidate.cost_usd:.5f}  '
              f'exact={score.candidate.exact}')
    print(f'  -> {decision.winner.name} ({decision.winner.kind})')
    print(f'  flip: {decision.flip_condition}')

### Why the parser wins — and how close it is

`flip_thresholds` solves for the exact score drop that hands the win over, per
axis. That number is what you go and measure.

In [ ]:
stable = choose_extraction('INV-1042', known_format=True)
print(stable.report())
print()
for finding in flip_thresholds(stable.ranked).findings:
    print(f"  axis {finding.criterion:16s} needs a drop of {finding.required_score_drop:.3f} "
          f"(feasible={finding.feasible}) -> {finding.narrative}")

print()
print('what-if replay:')
for check in sensitivity(stable, [
    Delta('a new vendor layout cuts parser accuracy (-0.20)', 'accuracy', 'regex-parser', score_delta=-0.20),
    Delta('double the accuracy weight', 'accuracy', 'regex-parser', weight_multiplier=2.0),
]):
    print(' ', check.reason)

## 2. Support-ticket routing — a classifier beats generation

Closed label set ⇒ classification, not generation: 5ms vs 900ms and ~0 per request
for ~5% less quality. Add a hard latency/cost budget and the choice stops being a
scoring question at all.

In [ ]:
routing = choose_task_type('Route this support ticket to the right team')
print(routing.report())
print()
print('closed-set signals found:', 'route' in routing.justification.lower())

print()
print('now impose p99 <= 50ms and cost <= $0/request:')
candidates = [score.candidate for score in routing.ranked.scores]
tight = decide(candidates, TASK_TYPE_CRITERIA, [latency_constraint(50.0), cost_constraint(0.0)])
print(tight.report())

print()
print('the same framework on an open-ended task:')
print(choose_task_type('Summarise this customer email in three friendly bullets').report())

## 3. FAQ answering — lookup table vs retrieval vs RAG

A fixed FAQ is a lookup table: exact, free, testable. Paraphrased questions need
embedding retrieval. Only genuine multi-document synthesis needs generation.

In [ ]:
scenarios = [
    ('40 fixed entries', dict(corpus_size=40)),
    ('40 entries, questions paraphrased', dict(corpus_size=40, question_variety='paraphrased')),
    ('5,000 entries, varied questions', dict(corpus_size=5000, question_variety='varied')),
    ('answers span several documents', dict(corpus_size=5000, question_variety='varied', answers_span_documents=True)),
]
for label, kwargs in scenarios:
    decision = choose_faq(**kwargs)
    ranking = ', '.join(f'{s.candidate.name} {s.total:.3f}' for s in decision.ranked.scores)
    print(f'{label:38s} -> {decision.winner.name:20s} [{ranking}]')
    print(f'{"":40s} flip: {decision.flip_condition}')

print()
print('a thin margin is a warning, not a victory:')
synthesis = choose_faq(corpus_size=5000, question_variety='varied', answers_span_documents=True)
rag = synthesis.ranked.by_name()['rag-generation'].total
retrieval = synthesis.ranked.by_name()['embedding-retrieval'].total
print(f'rag-generation {rag:.3f} vs embedding-retrieval {retrieval:.3f} '
      f'- a {rag - retrieval:.3f} margin: the weights decided this one')

## 4. The verifiability gate and the never-LLM list

Low verifiability + high stakes is the dangerous corner: it is a **block**, not a
warning. And some jobs are never an LLM's job.

In [ ]:
for label, candidate, stakes, exact in [
    ('unverifiable generation, high stakes', Candidate('rag', KIND_SMALL_LLM, verifiable=False), 'high', False),
    ('unverifiable generation, low stakes', Candidate('rag', KIND_SMALL_LLM, verifiable=False), 'low', False),
    ('verifiable LLM, high stakes', Candidate('extract', KIND_SMALL_LLM, verifiable=True), 'high', False),
    ('LLM asked for exactness', Candidate('totals', KIND_LARGE_LLM, verifiable=True), 'medium', True),
    ('deterministic code, high stakes', Candidate('policy', KIND_DETERMINISTIC, exact=True, verifiable=True), 'high', True),
]:
    verdict = verifiability_gate(candidate, stakes=stakes, requires_exactness=exact)
    print(f'{label:38s} allowed={verdict.allowed}')
    for warning in verdict.warnings:
        print(f'    [{warning.severity.upper()}] {warning.message}')

print()
for feature in ('arithmetic', 'authorization', 'cryptographic_exactness', 'legal_exactness'):
    print(f'{feature:26s} -> {never_llm_reason(feature)}')

## 5. Takeaways

- **Constraints first.** Hard requirements eliminate candidates; scores only rank
  what survived. A 3s p99 and $0.0001/request decide more features than capability does.
- **Deterministic-first.** If code, a parser, a lookup table or a classifier meets
  the requirement, it wins: cheaper, faster, exactly testable, no hallucination, no
  prompt-injection surface.
- **Verifiability is an axis.** High verifiability makes LLM generation safe (a
  verifier catches errors, and best-of-N/self-consistency works); low verifiability
  at high stakes is a block.
- **Classic flips:** regex beats LLM extraction on a known format; a classifier
  beats generation on a closed label set; a lookup table beats RAG on a fixed FAQ.
- **Always state the flip condition.** Measurable trigger + threshold + window, or
  the decision is just an opinion.
- The same axes return in topic 39: workflow vs agent is this decision applied to
  control flow.